# Experiment 5: Triton Latency & Profiling Variance Analysis

Validate the KBSS (Triton GPU kernel) subset by analyzing the coefficient of variation (CV) of latency measurements. Hardware latency is non-deterministic due to thermal throttling, clock fluctuations, and OS interrupts. High CV establishes an upper bound on any model's achievable accuracy.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy import stats

from utils.data_loader import load_partition, parse_metadata, print_mode_banner, is_validation_mode
from utils.plotting import setup_style, COLORS
import matplotlib.pyplot as plt
import seaborn as sns

VALIDATION_MODE = is_validation_mode()
print_mode_banner(VALIDATION_MODE)
setup_style()

## 1. Load KBSS (KernelBook) Partition

In [ ]:
df = load_partition('KBSS', validation_mode=VALIDATION_MODE)
print(f"Loaded {len(df):,} KBSS rows")
print(f"metric_type: {df['metric_type'].unique()}")
assert all(df['metric_type'] == 'latency_ms'), "Expected latency_ms metric!"
print("\u2705 All rows have metric_type = 'latency_ms'")
print(f"\nLatency (ms) summary:")
print(df['val_accuracy'].describe().to_string())

## 2. Parse Metadata: Extract stddev_ms

The KBSS metadata contains `{'stddev_ms': <value>}` — the standard deviation of repeated latency measurements for each kernel.

In [ ]:
df = parse_metadata(df)

if 'meta_stddev_ms' in df.columns:
    df['stddev_ms'] = pd.to_numeric(df['meta_stddev_ms'], errors='coerce')
    print(f"Successfully extracted stddev_ms for {df['stddev_ms'].notna().sum()} rows")
    print(f"\nstddev_ms summary:")
    print(df['stddev_ms'].describe().to_string())
else:
    print("\u26a0\ufe0f  stddev_ms not found in metadata. Checking raw metadata...")
    print(df['metadata'].head(3).to_string())
    # Try manual extraction
    import ast
    def extract_stddev(meta_str):
        try:
            d = ast.literal_eval(str(meta_str))
            return d.get('stddev_ms', np.nan)
        except:
            return np.nan
    df['stddev_ms'] = df['metadata'].apply(extract_stddev)
    print(f"Extracted stddev_ms for {df['stddev_ms'].notna().sum()} rows")

## 3. Compute Coefficient of Variation (CV)

CV = stddev_ms / latency_ms × 100%

CV tells us how much the latency measurement varies relative to its mean. High CV means the measurement itself is noisy.

In [ ]:
# Filter to rows where both latency and stddev are available and valid
df_valid = df.dropna(subset=['val_accuracy', 'stddev_ms']).copy()
df_valid = df_valid[df_valid['val_accuracy'] > 0]  # Avoid division by zero

df_valid['cv_percent'] = (df_valid['stddev_ms'] / df_valid['val_accuracy']) * 100

print(f"Rows with valid CV computation: {len(df_valid):,}")
print(f"\nCoefficient of Variation (CV%) summary:")
print(df_valid['cv_percent'].describe().to_string())

print(f"\nCV Distribution:")
print(f"  CV < 1%:   {(df_valid['cv_percent'] < 1).sum()} ({100*(df_valid['cv_percent'] < 1).mean():.1f}%)")
print(f"  CV < 5%:   {(df_valid['cv_percent'] < 5).sum()} ({100*(df_valid['cv_percent'] < 5).mean():.1f}%)")
print(f"  CV < 10%:  {(df_valid['cv_percent'] < 10).sum()} ({100*(df_valid['cv_percent'] < 10).mean():.1f}%)")
print(f"  CV >= 10%: {(df_valid['cv_percent'] >= 10).sum()} ({100*(df_valid['cv_percent'] >= 10).mean():.1f}%)")
print(f"  CV >= 20%: {(df_valid['cv_percent'] >= 20).sum()} ({100*(df_valid['cv_percent'] >= 20).mean():.1f}%)")

## 4. CV Distribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Coefficient of Variation (CV) of Triton Kernel Latency', 
             fontsize=16, fontweight='bold', color=COLORS['highlight'])

# Histogram
axes[0].hist(df_valid['cv_percent'], bins=50, color=COLORS['KBSS'], alpha=0.7, edgecolor='black')
axes[0].axvline(10, color=COLORS['primary'], linestyle='--', linewidth=2, label='CV = 10%')
axes[0].axvline(20, color='white', linestyle='--', linewidth=2, label='CV = 20%')
axes[0].set_title('CV Distribution (Histogram)', fontweight='bold')
axes[0].set_xlabel('CV (%)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.2)

# KDE
sns.kdeplot(df_valid['cv_percent'], ax=axes[1], color=COLORS['KBSS'], fill=True, alpha=0.3, linewidth=2)
axes[1].axvline(10, color=COLORS['primary'], linestyle='--', linewidth=2, label='CV = 10%')
axes[1].set_title('CV Distribution (KDE)', fontweight='bold')
axes[1].set_xlabel('CV (%)')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 5. Latency vs Standard Deviation Scatter

Color-coded by CV to identify which kernels have unstable measurements.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    df_valid['val_accuracy'], 
    df_valid['stddev_ms'],
    c=df_valid['cv_percent'],
    cmap='RdYlGn_r',
    s=30,
    alpha=0.7,
    edgecolors='none',
)

cbar = plt.colorbar(scatter, ax=ax, label='CV (%)')
ax.set_xlabel('Latency (ms)', fontsize=12)
ax.set_ylabel('Standard Deviation (ms)', fontsize=12)
ax.set_title('Latency vs Profiling Noise (colored by CV)', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.2)

# Add a reference line: CV = 10%
max_lat = df_valid['val_accuracy'].max()
x_line = np.linspace(0, max_lat, 100)
ax.plot(x_line, 0.10 * x_line, color='white', linestyle='--', alpha=0.5, label='CV = 10% line')
ax.legend()

plt.tight_layout()
plt.show()

print("\n\ud83d\udcca Points above the CV=10% line have high profiling noise.")
print("   These kernels' latencies vary significantly between runs.")

## 6. High-CV Kernel Analysis

Examine kernels with the highest measurement variance.

In [ ]:
high_cv = df_valid[df_valid['cv_percent'] >= 10].sort_values('cv_percent', ascending=False)

print(f"Kernels with CV >= 10%: {len(high_cv)} ({100*len(high_cv)/len(df_valid):.1f}% of total)")
print()

if len(high_cv) > 0:
    print("Top 10 highest-CV kernels:")
    print("="*80)
    for i, (_, row) in enumerate(high_cv.head(10).iterrows()):
        print(f"\n  #{i+1}: {row['identifier']}")
        print(f"    Latency: {row['val_accuracy']:.4f} ms")
        print(f"    Stddev:  {row['stddev_ms']:.4f} ms")
        print(f"    CV:      {row['cv_percent']:.1f}%")
        # Show first 100 chars of code
        code_preview = str(row['input'])[:100].replace('\n', ' ')
        print(f"    Code:    {code_preview}...")

## 7. Theoretical Error Floor

The CV establishes a theoretical lower bound on prediction error. No model can predict latency more accurately than the hardware's own measurement variance.

In [ ]:
mean_cv = df_valid['cv_percent'].mean()
median_cv = df_valid['cv_percent'].median()
p90_cv = np.percentile(df_valid['cv_percent'], 90)

print("="*80)
print("THEORETICAL ERROR FLOOR ANALYSIS")
print("="*80)
print()
print(f"Mean CV:   {mean_cv:.2f}%")
print(f"Median CV: {median_cv:.2f}%")
print(f"P90 CV:    {p90_cv:.2f}%")
print()
print("Interpretation:")
print(f"  If the hardware's own measurement variance is ~{median_cv:.1f}% (median),")
print(f"  then NO regression model (XGBoost, LLM, or otherwise) can achieve")
print(f"  prediction errors lower than ~{median_cv:.1f}% relative error.")
print()
if mean_cv > 10:
    print("  \u26a0\ufe0f HIGH VARIANCE: Mean CV > 10%")
    print("     This suggests significant profiling noise in the Triton benchmarks.")
    print("     Model evaluation metrics should account for this inherent noise floor.")
else:
    print("  \u2705 ACCEPTABLE VARIANCE: Mean CV <= 10%")
    print("     The profiling noise is within reasonable bounds for regression tasks.")

## Conclusion

In [ ]:
print("="*80)
print("EXPERIMENT 5 CONCLUSION")
print("="*80)
print()
print("KBSS (Triton kernel) latency profiling variance analysis:")
print(f"  Total kernels analyzed: {len(df_valid):,}")
print(f"  Mean CV: {mean_cv:.2f}%")
print(f"  Median CV: {median_cv:.2f}%")
print(f"  Kernels with CV >= 10%: {(df_valid['cv_percent'] >= 10).sum()} ({100*(df_valid['cv_percent'] >= 10).mean():.1f}%)")
print()
print("The Coefficient of Variation establishes the hardware noise floor:")
print(f"  Any model predicting Triton kernel latency cannot achieve better")
print(f"  than ~{median_cv:.1f}% relative error (median CV) on this dataset.")
print()
print("This validates the paper's acknowledgment of 'typical noise levels")
print("in regression evaluation' for low-level latency profiling.")
if VALIDATION_MODE:
    print("\n\u26a0\ufe0f  Note: Results from validation sample. Run full mode for complete analysis.")